# Kafka to Cassandra Streaming Test Notebook
This notebook processes real-time tracking data from Kafka and writes it to Cassandra.

In [1]:
import os
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from dotenv import load_dotenv

# Load environment variables from the root directory
load_dotenv(dotenv_path="../.env")

True

In [2]:
# Configuration
KAFKA_BOOTSTRAP_SERVERS = os.getenv('KAFKA_BOOTSTRAP_SERVERS', 'localhost:9092')
KAFKA_TOPIC = os.getenv('KAFKA_TOPIC', 'tracking_data')
CASSANDRA_HOST = os.getenv('CASSANDRA_HOST', 'localhost')
CASSANDRA_KEYSPACE = os.getenv('CASSANDRA_KEYSPACE', 'recruitment')
CASSANDRA_TABLE = os.getenv('CASSANDRA_TABLE', 'tracking')
CASSANDRA_JAR = os.getenv('CASSANDRA_JAR')

print(f"Kafka: {KAFKA_BOOTSTRAP_SERVERS}")
print(f"Cassandra: {CASSANDRA_HOST}")

Kafka: broker:29092
Cassandra: cassandra_etl


In [3]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("KafkaToCassandraStreamingTest") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.jars", CASSANDRA_JAR) \
    .config("spark.cassandra.connection.host", CASSANDRA_HOST) \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


In [4]:
# Define Schema for Kafka messages
schema = StructType([
    StructField("create_time", StringType(), True),
    StructField("bid", StringType(), True),
    StructField("campaign_id", IntegerType(), True),
    StructField("custom_track", StringType(), True),
    StructField("group_id", IntegerType(), True),
    StructField("job_id", IntegerType(), True),
    StructField("publisher_id", IntegerType(), True),
    StructField("ts", StringType(), True)
])

In [5]:
# 1. Read from Kafka
print(f"Reading from Kafka topic: {KAFKA_TOPIC}...")
df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "latest") \
    .load()

# 2. Parse JSON
parsed_df = df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*")

Reading from Kafka topic: tracking_data...


In [ ]:
# 3. Write to Cassandra
print(f"Streaming data to Cassandra table {CASSANDRA_KEYSPACE}.{CASSANDRA_TABLE}...")

query = parsed_df \
    .writeStream \
    .format("org.apache.spark.sql.cassandra") \
    .option("keyspace", CASSANDRA_KEYSPACE) \
    .option("table", CASSANDRA_TABLE) \
    .option("checkpointLocation", "/tmp/checkpoint_kafka_to_cassandra_test") \
    .outputMode("append") \
    .start()

try:
    query.awaitTermination()
except KeyboardInterrupt:
    print("Stopping Streaming Query...")
    query.stop()

Streaming data to Cassandra table recruitment.tracking...
